# 14 · Referencia walk-forward sin variables de régimen

En los notebooks 11–13 examinamos modelos con CPCV. Ahora construiremos una referencia que respete el orden cronológico: **aprender del pasado y predecir el año siguiente**. Servirá para comparar después qué aporta añadir información del entorno.

- **M0:** frecuencia de éxitos del entrenamiento de cada año.
- **M1:** logística L2 con retorno del último mes y momentum 12–1.

«Sin régimen» es una definición operativa: estas dos variables describen las señales y también pueden reflejar indirectamente el mercado. No se incorporan todavía volatilidad, retorno de 24 meses, drawdown ni interacciones.

**Entradas:** tablas de 01 → 02 → 07 → 08 y el entorno `.venv` con el extra `models`. No necesitas repetir los notebooks 09–13. El diseño, motivado por la guía del 15 de noviembre, está en [el protocolo](../docs/validacion_walk_forward.md). Se ha definido después de conocer los experimentos anteriores.

## 1. Un calendario que avanza año a año

El desarrollo sigue siendo 1927–1989. Para disponer de historia inicial, empezamos las pruebas en 1950. Aprendemos con la historia anterior y mantenemos los coeficientes fijos durante los doce meses de ese año. Al año siguiente ampliamos el entrenamiento y volvemos a ajustar. Son cuarenta periodos anuales de prueba, hasta 1989.

Aplicamos una purga de un mes objetivo anterior. Para probar 1950 usamos etiquetas hasta noviembre de 1949, diciembre de 1949 queda excluido. El calendario se define antes de retirar meses sin señal o sin características. Los meses futuros nunca entrenan ese ajuste. Como no hay entrenamiento posterior a su prueba, el embargo posterior de CPCV no elimina meses aquí.

La normalización causal de las características sí se actualiza con la historia anterior a cada nueva observación. Mantener los coeficientes durante el año no implica congelar en enero esa preparación.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from IPython.display import display

from tfm.data import project_root, save_result
from tfm.walk_forward import (FIRST_TEST_YEAR, LAST_TEST_YEAR, SIGNAL_COLUMNS,
    build_walk_forward_splits, load_walk_forward_inputs, summarize_walk_forward, run_walk_forward)

ROOT = project_root()
ejemplos, _, _ = load_walk_forward_inputs(ROOT)
particiones = build_walk_forward_splits()
cobertura = summarize_walk_forward(ejemplos, particiones)
display(cobertura.loc[cobertura.test_year.isin([FIRST_TEST_YEAR, LAST_TEST_YEAR])])

colores = {'train': ('Entrenamiento', '#9dc8b4'), 'purge': ('Purga', '#b74e40'),
           'validation': ('Prueba del año', '#244b70'), 'future': ('Futuro excluido', '#eeeeee')}
matriz = particiones.pivot(index='test_year', columns='target_period', values='role')
codigos = {nombre: i for i, nombre in enumerate(colores)}
fig, ax = plt.subplots(figsize=(12, 6))
ax.imshow(matriz.replace(codigos).to_numpy(dtype=int), aspect='auto', interpolation='nearest',
          cmap=ListedColormap([color for _, color in colores.values()]), vmin=-.5, vmax=3.5)
x = [i for i, mes in enumerate(matriz.columns) if mes.endswith('-01') and int(mes[:4]) % 10 == 0]
y = list(range(0, len(matriz), 5))
ax.set_xticks(x, [matriz.columns[i][:4] for i in x])
ax.set_yticks(y, [matriz.index[i] for i in y])
ax.set(title='Historia disponible para cada año de prueba', xlabel='Mes objetivo del calendario', ylabel='Año de prueba')
fig.legend(handles=[Patch(color=color, label=nombre) for nombre, color in colores.values()],
           loc='lower center', ncol=4, fontsize=9)
fig.tight_layout(rect=(0, .06, 1, 1))
plt.show()

## 2. Qué ejemplos usamos y qué aprende M1

Una etiqueta indica si la operación aislada de la regla, neta de costes y dividendos aproximados, superó al efectivo. Solo tienen etiqueta las señales activas. Un mes sin operación no es un fallo del filtro y no recibe probabilidad cero.

Conservamos la muestra común de los experimentos anteriores: las diez características preparadas deben estar disponibles tras su calentamiento. M1 solo recibe **dos columnas explícitas**: `z_price_return_1m` y `z_momentum_12_1`. Las restantes no se utilizan para predecir. Así las comparaciones posteriores tendrán las mismas fechas, en lugar de ampliar la muestra de cada modelo por separado.

Usamos la misma logística regularizada del 11, con C=1, sin ponderar clases ni buscar parámetros. Las dos reglas se ajustan por separado. Los años sin señales permanecen en la cobertura. Si solo hay una clase de prueba, la log-loss se puede calcular y el AUC no.

In [ ]:
print('Predictores de M1:', SIGNAL_COLUMNS)
display(cobertura.groupby('strategy').agg(
    ejemplos_iniciales=('train_examples', 'min'),
    ejemplos_prueba=('validation_examples', 'sum'),
    minimo_anual=('validation_examples', 'min'),
    maximo_anual=('validation_examples', 'max')))
display(cobertura.loc[cobertura.validation_classes.lt(2),
    ['test_year', 'strategy', 'validation_calendar_months', 'no_signal_months',
     'excluded_active_months', 'validation_examples', 'validation_classes']])

## 3. Ajustar con el pasado y predecir una sola vez

M0 asigna a cada ejemplo del año la proporción de etiquetas positivas de su entrenamiento. M1 estima una probabilidad a partir de las dos características. Ninguno utiliza las etiquetas del año probado para ajustarse.

A diferencia del resumen de CPCV, cada ejemplo recibe **una única predicción cronológica** de cada modelo. No se promedian cinco modelos ni se cuentan repetidamente los mismos resultados. Si un ajuste no es estimable, sus probabilidades quedan ausentes, no se eliminan esos meses para mejorar el resumen.

In [ ]:
def mostrar_progreso(completadas, total):
    if completadas == 1 or completadas % 5 == 0:
        print(f'Años evaluados: {completadas}/{total}', flush=True)

tablas, informe = run_walk_forward(ROOT, progress=mostrar_progreso)
resumen = tablas['walk_forward_summary']
predicciones = tablas['walk_forward_predictions']
metricas = tablas['walk_forward_metrics']
coeficientes = tablas['walk_forward_coefficients']
print('Estado del recorrido:', informe['experiment_status'])
display(resumen[['test_year', 'strategy', 'train_first_target', 'train_last_target',
                'train_examples', 'validation_examples', 'train_classes', 'validation_classes', 'status']])

## 4. Leer los resultados

**AUC** describe si las etiquetas favorables reciben probabilidades más altas que las desfavorables. **Log-loss** penaliza probabilidades equivocadas, especialmente las muy seguras. Menor log-loss es favorable, mayor AUC indica mejor ordenación. Ninguna mide por sí sola rentabilidad.

El resumen principal (`pooled`) reúne las predicciones de prueba de los cuarenta años por regla y calcula la métrica una sola vez. No es una media de las AUC anuales. M0 tiene una probabilidad constante dentro de cada año, pero puede cambiar al reentrenarse: su AUC reunida puede diferir de 0,5.

Mostramos también cada año. `single_class` significa que no podemos calcular AUC porque todas las etiquetas son iguales, `no_examples`, que no hubo ejemplos. Estos años permanecen en las tablas. Si faltase una predicción necesaria, el resumen correspondiente quedaría no disponible.

In [ ]:
display(metricas.loc[metricas.aggregation.eq('pooled'),
    ['strategy', 'model', 'examples', 'positive', 'negative', 'log_loss', 'auc', 'status', 'auc_status']])
anuales = metricas.loc[metricas.aggregation.eq('year')]
display(anuales[['test_year', 'strategy', 'model', 'examples', 'log_loss', 'auc', 'status', 'auc_status']])
display(anuales.loc[anuales.model.eq('M1')].groupby(['strategy', 'auc_status']).size().rename('Años').reset_index())

## 5. Ver las probabilidades a lo largo del tiempo

Las líneas muestran las estimaciones de M0 y M1. Los puntos permiten ver también los meses aislados. Los huecos son meses sin ejemplo utilizable: no los rellenamos ni conectamos mediante interpolación. Una probabilidad alta significa que el modelo considera más probable superar al efectivo según nuestra etiqueta, no representa el tamaño de una ganancia ni una recomendación de inversión.

In [ ]:
import matplotlib.dates as mdates

meses_prueba = pd.period_range(f'{FIRST_TEST_YEAR}-01', f'{LAST_TEST_YEAR}-12', freq='M').astype(str)
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True, sharey=True)
for ax, regla in zip(axes, ('momentum', 'reversal')):
    serie = predicciones.loc[predicciones.strategy.eq(regla)].set_index('target_period').reindex(meses_prueba)
    fechas = pd.PeriodIndex(serie.index, freq='M').to_timestamp()
    ax.plot(fechas, serie.probability_naive, label='M0 · frecuencia pasada', color='#c77832', linewidth=1.2, marker='.', markersize=2)
    ax.plot(fechas, serie.probability_logistic, label='M1 · información de las señales', color='#244b70', linewidth=1, marker='.', markersize=2)
    ax.set(title=regla, ylabel='Probabilidad estimada', ylim=(0, 1))
    ax.grid(alpha=.2)
axes[0].legend(loc='upper left', fontsize=8)
axes[-1].set_xlabel('Mes objetivo de prueba')
axes[-1].xaxis.set_major_locator(mdates.YearLocator(10))
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
fig.tight_layout()
plt.show()

## 6. Alcance y continuación

Este notebook establece la referencia para estudiar después si las variables del entorno aportan una mejora adicional. No selecciona variables, incorpora regímenes, aplica SHAP ni simula decisiones de cartera. Las comparaciones con M2/M3, las sombras específicas y la evaluación económica siguen pendientes. Las sombras de la logística de diez características no validan automáticamente este modelo.

El orden temporal evita entrenar con resultados posteriores al año de prueba. Sin embargo, seguimos usando precios medios mensuales de Shiller, dividendos interpolados y versiones actuales de las fuentes. La purga no certifica sus fechas históricas de publicación. Es una aproximación retrospectiva, no acredita operaciones ejecutables ni disponibilidad point-in-time. El periodo desde 1990 permanece reservado.

La siguiente celda guarda cinco tablas con sus JSON, conservando los resultados anteriores. El historial del intento está en `outputs/experiments/walk_forward_trials.jsonl`. Si falla el guardado, repite la celda completa. Antes del commit, limpia las salidas y guarda.

Referencia: guía del tutor del 15 de noviembre de 2026, apartado 5, pp. 19–21. La adaptación, el calendario y los supuestos están en [el protocolo](../docs/validacion_walk_forward.md). Se reutilizan las metodologías y referencias de preparación, etiquetado y logística ya documentadas en el proyecto.

In [ ]:
guardados = []
for nombre, tabla in tablas.items():
    salida = save_result(ROOT, nombre, tabla, informe)
    guardados.append({'Tabla': nombre + '.parquet', 'Filas': salida['rows']})
display(pd.DataFrame(guardados))
print('Historial:', informe['trial_log'])
print('Evaluación pendiente:', informe['skill_gate_status'])